In [ ]:
import squidpy as sq
import scanpy as sc
import scipy as scp
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import scanorama
from pathlib import Path
import sklearn
import seaborn as sns

In [ ]:
sc.logging.print_versions()
sc.set_figure_params(facecolor="white", figsize=(8, 8))
sc.settings.verbosity = 3

In [ ]:
sclcSC = sc.read_h5ad('data/sclcHV.h5ad')
# sclcVisium = sc.read_visium(path='visium/', count_file='sclcVisiumRaw.h5') #check path and set image path
sclcVisiumFiltered = sc.read_visium(path='visium/', count_file='sclcVisiumFiltered.h5', )

In [ ]:
sclcSC.obs.cell_type

In [ ]:
sc.pp.calculate_qc_metrics(sclcVisiumFiltered, inplace=True)

In [ ]:
sclcVisiumFiltered.var.index

In [ ]:
sclcVisiumFiltered.var_names_make_unique()
sc.pl.violin(sclcVisiumFiltered, ['n_genes_by_counts', 'total_counts'], jitter=0.4, multi_panel=True)

In [ ]:
integration = scanorama.correct_scanpy([sclcSC,sclcVisiumFiltered], return_dimred=True)

In [ ]:
sclcSC.obsm

In [ ]:
sclcData = sc.concat(
    integration,
    label="dataset",
    keys=["single-cell", "spatial"],
    join="outer",
    uns_merge="first",
)

In [ ]:
sclcData

In [ ]:
from sklearn.metrics.pairwise import cosine_distances

distance = 1 - cosine_distances(
    sclcData[sclcData.obs.dataset == "single-cell"].obsm[
        "X_scanorama"
    ],
    sclcData[sclcData.obs.dataset == "spatial"].obsm[
        "X_scanorama"
    ],
)

In [ ]:
def label_transfer(dist, labels):
    '''
    Probability based labels' assignment function
    '''
    lab = pd.get_dummies(labels).to_numpy().T
    class_prob = lab @ dist
    norm = np.linalg.norm(class_prob, 2, axis=0)
    class_prob = class_prob / norm
    class_prob = (class_prob.T - class_prob.min(1)) / class_prob.ptp(1)
    return class_prob

In [ ]:
class_prob_sclc = label_transfer(distance, sclcSC.obs.cell_type) # compute labels

In [ ]:
cp_sclc_df = pd.DataFrame(
    class_prob_sclc,
    columns=sorted(sclcSC.obs["cell_type"].cat.categories),
)

cp_sclc_df.index = sclcVisiumFiltered.obs.index  # transfer index

In [ ]:
sclc_transfer = sclcVisiumFiltered.copy()
sclc_transfer.obs = pd.concat(
    [sclcVisiumFiltered.obs, cp_sclc_df], axis=1
)

In [ ]:
sclc_transfer.var_names

In [ ]:
sclc_transfer

In [ ]:
sc.pl.spatial(
    sclc_transfer,
    img_key="hires",
    color=['B cell', 'CD1c-positive myeloid dendritic cell', 'CD4-positive, alpha-beta T cell', 'CD8-positive, alpha-beta T cell', 'T cell', 'alveolar macrophage', 'alveolar type 1 fibroblast cell', 'alveolar type 2 fibroblast cell', 'bronchus fibroblast of lung', 'capillary endothelial cell', 'ciliated columnar cell of tracheobronchial tree', 'classical monocyte', 'club cell', 'conventional dendritic cell', 'dendritic cell', 'elicited macrophage', 'endothelial cell of lymphatic vessel', 'epithelial cell of alveolus of lung', 'epithelial cell of lower respiratory tract', 'lung macrophage', 'lung pericyte', 'mast cell', 'mesothelial cell', 'nasal mucosa goblet cell', 'native cell', 'natural killer cell', 'non-classical monocyte', 'plasma cell', 'plasmacytoid dendritic cell', 'pulmonary artery endothelial cell', 'respiratory basal cell', 'respiratory hillock cell', 'tracheobronchial smooth muscle cell', 'type I pneumocyte', 'type II pneumocyte', 'vein endothelial cell'],
    size=1.5,
)

In [ ]:
sc.pl.spatial(
    sclc_transfer,
    img_key="hires",
    color=['in_tissue', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts'],
    size=1.5,
)

In [ ]:
# List of cell type columns
cell_type_columns = ['B cell', 'CD1c-positive myeloid dendritic cell', 'CD4-positive, alpha-beta T cell', 'CD8-positive, alpha-beta T cell', 'T cell', 'alveolar macrophage', 'alveolar type 1 fibroblast cell', 'alveolar type 2 fibroblast cell', 'bronchus fibroblast of lung', 'capillary endothelial cell', 'ciliated columnar cell of tracheobronchial tree', 'classical monocyte', 'club cell', 'conventional dendritic cell', 'dendritic cell', 'elicited macrophage', 'endothelial cell of lymphatic vessel', 'epithelial cell of alveolus of lung', 'epithelial cell of lower respiratory tract', 'lung macrophage', 'lung pericyte', 'mast cell', 'mesothelial cell', 'nasal mucosa goblet cell', 'native cell', 'natural killer cell', 'non-classical monocyte', 'plasma cell', 'plasmacytoid dendritic cell', 'pulmonary artery endothelial cell', 'respiratory basal cell', 'respiratory hillock cell', 'tracheobronchial smooth muscle cell', 'type I pneumocyte', 'type II pneumocyte', 'vein endothelial cell']

# Create a new column 'cell_type' by combining existing cell type columns
sclc_transfer.obs['cell_type'] = pd.Categorical(sclc_transfer.obs[cell_type_columns].idxmax(axis=1))

# Plot all cells with the newly created 'cell_type' variable
sc.pl.spatial(
    sclc_transfer,
    img_key="hires",
    color='cell_type',
    groups=cell_type_columns,
    size=1
)

# # plotted by inferring maximum expression
# sclc_transfer.obs[cell_type_columns].idxmax(axis=1) finds the column name (cell type) with the maximum expression for each cell.
# pd.Categorical() converts the result into a categorical variable, which will be stored in the new cell_type column.
# The spatial plot is then created using this newly created cell_type variable as the color variable.

In [ ]:
sc.pl.spatial(
    sclc_transfer,
    img_key="hires",
    color=['AGER','CAV1'],
    size=1
)
sc.pl.spatial(
    sclc_transfer,
    img_key="hires",
    color=['cell_type'],
    groups=['type I pneumocyte'],
    size=1
)